# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an interactive, step-by-step guide for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and provides mass spectrometry-based proteomics data from human mast cell extracellular vesicles.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their corresponding `@id` fields.

The Croissant schema organizes data into one or more record sets. Each record set defines a table-like structure consisting of fields. All identifiers shown are the canonical `@id` values, which you should use in code for extraction and referencing.

In [ ]:
# List all record sets with `@id` and their fields
print('Available Record Sets in the Dataset:')
record_sets = []
for rset in dataset.metadata.record_sets:
    print(f"  - @id: {rset['@id']}, Name: {rset.get('name','no name')}")
    record_sets.append(rset['@id'])
    if 'fields' in rset:
        print('    Fields:')
        for field in rset['fields']:
            print(f"      * @id: {field['@id']} (name: {field.get('name', field['@id'])}) | type: {field.get('dataType','')}")
    else:
        print('    No fields found.')
# Show an example record from each record set
for rset_id in record_sets:
    print(f'Example record from record set {rset_id}:')
    try:
        record_iter = dataset.records(record_set=rset_id)
        print(next(record_iter))
    except Exception as e:
        print(f'  Could not fetch records for {rset_id}: {e}')

## 3. Data Extraction
Load data from a selected record set into a Pandas DataFrame for analysis. Record sets and field `@id`s are referenced directly from the Croissant metadata.

We'll extract *all* available record sets.

In [ ]:
# Extract all record set dataframes
dataframes = {}
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for {record_set_id} with columns:")
        print(dataframes[record_set_id].columns.tolist())
        print(dataframes[record_set_id].head(2))
        print()
    except Exception as e:
        print(f'Could not load DataFrame for {record_set_id}: {e}')

## 4. Exploratory Data Analysis (EDA)
Choose a numeric field for analysis from one of the record sets. We'll demonstrate basic filtering, normalization, and grouping operations using the field and record set `@id`s.

**Note:** Replace the examples below with actual `@id` values from earlier cells matching appropriate numeric and grouping fields.

In [ ]:
# Example: Use a record set and numeric field by @id
# Fill these variables with your selection from the 'record_sets' cell (example values below; edit as needed)
record_set_id = record_sets[0]  # Change index as appropriate
df = dataframes[record_set_id]

# Display numeric fields to help user select one
print("Available columns in selected record set:", df.columns.tolist())

# User: choose a numeric field -- edit below as appropriate
numeric_field = df.select_dtypes(include=[float, int]).columns[0] if len(df.select_dtypes(include=[float, int]).columns)>0 else df.columns[0]
print(f"Using numeric field: {numeric_field}")

# Filtering: Keep rows where numeric_field > threshold
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalization: Z-score
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping: Use a string/categorical field if possible
group_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
group_field = group_candidates[0] if group_candidates else None
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped data by {group_field} (mean of {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships in the filtered data using fields accessible via their `@id` identifiers. Example: histogram of the numeric field and boxplot by grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Plot histogram of the normalized numeric field
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[f"{numeric_field}_normalized"], bins=30, kde=True)
plt.title(f"Distribution of normalized {numeric_field}")
plt.xlabel(f"{numeric_field} (z-score normalized)")
plt.show()

# If group_field exists, plot a boxplot by category
if group_field and group_field in filtered_df.columns:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and examine a dataset described by the FAIR² Croissant schema using the `mlcroissant` library.

**Summary:**
- Explored the available record sets and fields by their canonical `@id`s.
- Extracted data from all record sets into Pandas DataFrames.
- Performed basic filtering, normalization, and grouping on a chosen numeric field.
- Visualized data distributions and comparisons by category.

**Next steps:**
Continue your own scientific, statistical, or ML analysis by adapting field and record set selections and applying domain-specific data transformations.